# Destiny — Tesla T4 Optimization Lab

**Independent lab, not an episode renderer.** No Drive mount by default, no voice/audio models, no npm/Remotion install, no full video export. Approved models, lighting, geometry and all 19 shot definitions stay unchanged. Work is isolated in **`/content/destiny-gpu-lab/`**. This storage disappears on runtime reset: **download the report ZIP**.

Choose a fresh **T4 GPU runtime**. O1 prepares six supported configurations; O2 runs them sequentially; O3 reviews paired timings/images; optional O4 exports settings only after explicit approval. Fast mode: **18 real frames** (three shots × six cases × one trial), unsupported cases skipped with reasons. Standard: **108 real frames**, three consecutive frames per shot, two reversed-order trials, for temporal review. No 2672-frame action exists.

The user-reported M3 trial (19 new / 11 cached) spent 285.40 / 294.32 s in Cycles+compositing. That isolates the broad operation, not exclusive GPU kernels or a cold speedup. CPU denoising/compositing defaults are hypotheses to test; low GPU utilization alone is not a configuration diagnosis.


In [ ]:
# @title O1. Install the minimal lab and prepare a NEW benchmark
run_id = 't4-fast-01' # @param {type:"string"}
benchmark_mode = 'fast' # @param ['fast','standard']
resolution = '540×960' # @param ['360×640','540×960']
restore_source_commit = '' # @param {type:"string"}
import os, sys, json, hashlib, shutil, subprocess, tarfile, urllib.request, re, signal
from pathlib import Path
from IPython.display import display, HTML, Image
LAB_ROOT=Path('/content/destiny-gpu-lab')
assert LAB_ROOT.resolve()==LAB_ROOT and not LAB_ROOT.is_mount(), 'Lab root must be a dedicated local directory.'
LAB_ROOT.mkdir(exist_ok=True)
marker=LAB_ROOT/'.destiny-gpu-lab.json'
if marker.exists(): assert json.loads(marker.read_text())['kind']=='destiny-gpu-lab', 'Foreign workspace.'
else: marker.write_text(json.dumps({'kind':'destiny-gpu-lab','version':1}))
REPO=LAB_ROOT/'repo';TOOLS=LAB_ROOT/'tools';TOOLS.mkdir(exist_ok=True)
BRANCH='feat/atheon-performance-cinematic-direction'
def cmd(args,**kw): subprocess.run([str(x) for x in args],check=True,**kw)
if not REPO.exists():
    cmd(['git','clone','--depth','1','--filter=blob:none','--no-checkout','--single-branch','--branch',BRANCH,'https://github.com/Alexanst54/Destiny-Explained-Studio.git',REPO])
    cmd(['git','-C',REPO,'sparse-checkout','set','--no-cone','/remotion/blender/','/remotion/studio/','/remotion/atlas/universal/catalog.json','/episodes/atheon-explained-en/script.json','/episodes/atheon-explained-en/audio-evidence/timing.json'])
    cmd(['git','-C',REPO,'checkout',BRANCH])
else:
    assert (REPO/'.git').exists(), 'Existing foreign repo directory; use a fresh runtime.'
    assert not subprocess.check_output(['git','-C',REPO,'status','--porcelain'],text=True).strip(), 'Edited checkout: preserve it and start a fresh runtime.'
if restore_source_commit:
    assert re.fullmatch('[0-9a-fA-F]{40}',restore_source_commit), 'Provide the exact 40-character recorded source commit.'
    cmd(['git','-C',REPO,'fetch','--depth','1','origin',restore_source_commit])
    cmd(['git','-C',REPO,'checkout','--detach',restore_source_commit])
# No Torch/Kokoro, narration WAVs, browser, npm or production exports are installed.
BLENDER=TOOLS/'blender-4.3.2-linux-x64/blender'
if not BLENDER.exists():
    base='https://download.blender.org/release/Blender4.3/'
    filename='blender-4.3.2-linux-x64.tar.xz';archive=TOOLS/filename
    checksum_text=urllib.request.urlopen(base+'blender-4.3.2.sha256').read().decode()
    expected=next(line.split()[0] for line in checksum_text.splitlines() if line.split()[-1].lstrip('*')==filename)
    urllib.request.urlretrieve(base+filename,archive)
    with archive.open('rb') as f:
        h=hashlib.sha256()
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    assert h.hexdigest()==expected, 'Official Blender checksum mismatch.'
    with tarfile.open(archive) as tf:
        for member in tf.getmembers():
            assert not Path(member.name).is_absolute() and '..' not in Path(member.name).parts, 'Unsafe archive member.'
        tf.extractall(TOOLS,filter='data')
    archive.unlink() # Only this disposable lab download, never any Drive file.
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    cmd(['apt-get','-qq','update']);cmd(['apt-get','-qq','install','-y','ffmpeg'])
WIDTH,HEIGHT=map(int,resolution.split('×'))
LAB_SCRIPT=REPO/'remotion/blender/gpu_lab/lab.py'
assert LAB_SCRIPT.exists(), 'This source predates the standalone lab. Start a fresh runtime for latest code.'
def lab_action(action,extra=()):
    args=[sys.executable,str(LAB_SCRIPT),action,'--root',str(LAB_ROOT),'--run-id',run_id,'--mode',benchmark_mode,'--width',str(WIDTH),'--height',str(HEIGHT),'--blender',str(BLENDER)]+list(extra)
    p=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    try:
        for line in p.stdout:print(line.rstrip(),flush=True)
        if p.wait():raise RuntimeError('Lab action failed. Read preserved local logs; no episode was touched. Resume unchanged settings/source or choose a NEW run ID.')
    except BaseException:
        try:os.killpg(p.pid,signal.SIGINT)
        except ProcessLookupError:pass
        try:p.wait(timeout=25)
        except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
        raise
RUN=LAB_ROOT/'runs'/run_id
lab_action('prepare')
print('Source commit:',subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())
print('Disk before benchmark:',shutil.disk_usage(LAB_ROOT))
print('WARNING: /content is temporary. Download the final ZIP before resetting Colab. Google Drive is NOT mounted by this notebook.')


In [ ]:
# @title O2. Run the paired benchmark on the actual Tesla T4
run_t4_optimization_benchmark = False # @param {type:"boolean"}
assert run_t4_optimization_benchmark, 'Enable this bounded benchmark after selecting a T4 runtime.'
assert 'LAB_SCRIPT' in globals(), 'Run O1 first.'
lab_action('run')
lab_action('report') # Preserves comparison images/reports/logs, ZIP; cleans only marked lab intermediates after successful completion.
print('Disk after benchmark/report:',shutil.disk_usage(LAB_ROOT))
print('Results:',RUN/'final/index.html')
print('One report ZIP:',RUN/'destiny-gpu-lab-results.zip')
print('Failures preserve intermediates for local resume; reset loses unsaved work. No Drive writes occurred.')


In [ ]:
# @title O3. Review performance, paired images and download ONE ZIP
review_run_id = '' # @param {type:"string"}
comparison_case = 'candidate-48' # @param ['samples-32','samples-24','alternate-denoiser-48','candidate-48','candidate-32']
comparison_shot = 'mechanism' # @param ['vault-establish','past-gate','mechanism']
comparison_trial = 1 # @param [1,2] {type:"raw"}
comparison_frame = 0 # @param [0,1,2] {type:"raw"}
show_difference_map = True # @param {type:"boolean"}
download_report_zip = True # @param {type:"boolean"}
import html
REVIEW=LAB_ROOT/'runs'/(review_run_id or run_id)
assert (REVIEW/'final/results.json').exists(), 'Run O2 or explicitly restore a report ZIP first.'
report=json.loads((REVIEW/'final/results.json').read_text())
rows=''.join('<tr>'+''.join('<td style="padding:8px">'+html.escape(str(c[k]))+'</td>' for k in ['case','status','medianInvocationSeconds','speedRatio','eligibleForProductionReview'])+'</tr>' for c in report['comparisons'])
display(HTML('<div style="overflow:auto"><table><tr><th>Case</th><th>Status</th><th>Wall seconds</th><th>Cold T4 ratio</th><th>Review eligible</th></tr>'+rows+'</table></div>'))
print(report['recommendation'])
for c in report['comparisons']:print(c['case'],c['projection'])
prefix=REVIEW/f'final/images/trial-{comparison_trial}'/comparison_case/f'{comparison_shot}-{comparison_frame}'
pair=Path(str(prefix)+'-pair.png');difference=Path(str(prefix)+'-diff.png')
if pair.exists():
    print('Baseline LEFT / candidate RIGHT');display(Image(filename=str(pair),width=600))
    if show_difference_map:print('Difference ×4, not a visual-equivalence verdict');display(Image(filename=str(difference),width=300))
else:print('No image for that selection: feature may be unsupported/failed, or fast mode has frame 0 / trial 1 only.')
print('Full HTML, raw telemetry, JSON, PSNR/SSIM and temporal residuals are in the ZIP. Fast mode cannot validate flicker.')
if download_report_zip:
    from google.colab import files
    files.download(str(REVIEW/'destiny-gpu-lab-results.zip'))


In [ ]:
# @title O4. OPTIONAL — export reviewed settings, never modify production
selected_measured_case = 'candidate-48' # @param ['samples-32','samples-24','alternate-denoiser-48','candidate-48','candidate-32']
approve_performance_and_visual_quality = False # @param {type:"boolean"}
export_settings_for_new_project = False # @param {type:"boolean"}
assert export_settings_for_new_project and approve_performance_and_visual_quality, 'Explicit approval is required; nothing will be applied to existing projects.'
assert 'REVIEW' in globals(), 'Review O3 performance and images first.'
EXPORT_RUN=review_run_id or run_id
cmd([sys.executable,LAB_SCRIPT,'export','--root',LAB_ROOT,'--run-id',EXPORT_RUN,'--selected-case',selected_measured_case,'--approved'])
print('Exported recipe for a NEW cinematic project only. Production notebook, models, narration and caches are unchanged. No episode render was launched.')
from google.colab import files
files.download(str(LAB_ROOT/'runs'/EXPORT_RUN/'final/recommended-config.json'))


## Reading the results

Cold means **fresh frame caches**, not an empty global driver/compiler cache: small feature probes run before trials. Standard trials reverse case order to reduce order bias; inspect the first frame and steady-state frames separately. A warm/partial case cannot produce a cold speedup. A completed original cold trial can be reused as archived evidence after CRC/SHA validation; it is labeled as such.

Only one Blender process runs at a time. CPU devices/duplicate CUDA entries are logged and disabled for all cases; the selected physical T4/backend is explicit. Denoiser and GPU compositor features require a successful 32×32 render probe, not just an RNA enum. Unsupported cases/features are skipped with reasons; GPU failures/OOM never silently retry on CPU.

No lighting, volume density/step size, bounces, shadow, adaptive threshold, seed, geometry, pose or camera changes are permitted between configurations. Candidate settings only change supported GPU denoising/compositing scheduling and auto-tiling. Those changes are **experiments, not recommended defaults**. PSNR/SSIM and residual changes do not prove visual quality; review metal, glow, portals, mist, shadows and consecutive-frame flicker.

Render-time projections are coarse three-shot **Cycles+compositor only** estimates; complete production time remains unknown because Drive checkpointing, narration, Remotion and export are intentionally absent. There is no six-hour promise. O4 rejects fast-only, CPU, warm, failed, source-mismatched or non-improving evidence, and requires two repeated cold T4 trials (median gain ≥5%, each trial >2%) plus explicit human quality approval. It exports a JSON recipe; it does not integrate it automatically.

After successful reporting, disposable intermediate PNG/cache directories are deleted **only inside the marked lab run**. Final comparison images, reports, raw logs and ZIP remain. Failed jobs keep local intermediates for resume; an abrupt runtime reset loses unsaved files. Download first.


In [ ]:
# @title OPTIONAL Drive — save ONE report ZIP or restore an explicitly chosen lab ZIP
save_selected_zip_to_drive = False # @param {type:"boolean"}
restore_selected_zip_from_drive = False # @param {type:"boolean"}
chosen_lab_zip = '' # @param {type:"string"}
restore_name = 'saved-01' # @param {type:"string"}
if save_selected_zip_to_drive or restore_selected_zip_from_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    REPORTS=Path('/content/drive/MyDrive/Destiny-GPU-Lab-Reports')
    if save_selected_zip_to_drive:
        assert re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9_-]{0,70}',run_id), 'Invalid report name.'
        source=RUN/'destiny-gpu-lab-results.zip'
        assert source.exists(), 'Generate/download the final lab report first.'
        suffix=hashlib.sha256(source.read_bytes()).hexdigest()[:12]
        REPORTS.mkdir(exist_ok=True)
        destination=REPORTS/(run_id+'-'+suffix+'.zip')
        assert not destination.exists(), 'This report already exists; no overwrite is allowed.'
        shutil.copyfile(source,destination);print('Only this final ZIP saved:',destination)
    if restore_selected_zip_from_drive:
        selected=Path(chosen_lab_zip)
        assert '..' not in selected.parts and selected.is_relative_to(REPORTS) and selected.suffix=='.zip', 'Choose an explicit lab ZIP inside Destiny-GPU-Lab-Reports; episode folders are never inspected.'
        uploaded=LAB_ROOT/'selected-report.zip';assert not uploaded.exists(), 'Use a new runtime or a different explicit local upload file; no overwrite.'
        shutil.copyfile(selected,uploaded)
        cmd([sys.executable,LAB_SCRIPT,'restore','--root',LAB_ROOT,'--run-id',restore_name,'--archive',uploaded])
        print('To review, set O3 review_run_id to: restored-'+restore_name)
else:print('Drive remains unmounted. Use the O3 download ZIP button for minimal storage.')
